In [ ]:
# pip install pandas folium geopy 


import pandas as pd
import folium
from folium.plugins import MarkerCluster
from pathlib import Path


# ==========================================
# 1. 파일 경로 설정
# ==========================================

DATA_DIR = Path("data")

FILE_2020 = DATA_DIR / "parking_2020.csv"
FILE_2024 = DATA_DIR / "parking_violation_2024.csv"
FILE_2025 = DATA_DIR / "parking_violation_2026.csv"


# ==========================================
# 2. CSV 파일 불러오기
# ==========================================

def load_csv(file_path, encodings=("utf-8", "cp949", "euc-kr")):
    """여러 인코딩을 시도하여 CSV 파일을 불러온다."""

    for encoding in encodings:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"\n[{file_path.name}] 불러오기 성공: {encoding}")
            print(f"데이터 크기: {df.shape}")
            print("컬럼:", df.columns.tolist())
            return df

        except UnicodeDecodeError:
            continue

    raise ValueError(f"{file_path.name}의 인코딩을 확인할 수 없습니다.")


# ==========================================
# 3. 2020년 데이터 전처리
# ==========================================

def clean_2020(df):
    """2020년 좌표 데이터를 공통 형식으로 변환한다."""

    result = pd.DataFrame()

    # 컬럼명 확인 후 실제 파일에 맞게 수정할 것
    result["위도"] = pd.to_numeric(df["위도"], errors="coerce")
    result["경도"] = pd.to_numeric(df["경도"], errors="coerce")

    result["장소"] = df["행정동"].astype(str)
    result["발생일"] = pd.to_datetime(
        df["기준일자"], errors="coerce"
    )

    result["시간대"] = (
        df["위반상태"]
        .astype(str)
        .str.extract(r"(주간|야간)", expand=False)
    )

    result["출처"] = "2020년"

    result = result.dropna(subset=["위도", "경도"])

    return result


# ==========================================
# 4. 2024년 데이터 전처리
# ==========================================

def clean_2024(df):
    """2024년 장소명 및 시간 데이터를 공통 형식으로 변환한다."""

    result = pd.DataFrame()

    result["장소"] = df["위반장소"].astype(str).str.strip()

    result["발생일"] = pd.to_datetime(
        df["위반일자"], errors="coerce"
    )

    # 시간 데이터에서 시(hour) 추출
    time = pd.to_datetime(
        df["위반시간"].astype(str),
        format="%H:%M:%S",
        errors="coerce"
    )

    # 시간대 구분 기준: 06:00~17:59 주간, 그 외 야간
    result["시간대"] = time.dt.hour.map(
        lambda hour: (
            "주간" if 6 <= hour < 18 else "야간"
        ) if pd.notna(hour) else None
    )

    # 아직 좌표가 없으므로 빈 값으로 설정
    result["위도"] = pd.NA
    result["경도"] = pd.NA

    result["출처"] = "2024년"

    return result


# ==========================================
# 5. 2025년 데이터 전처리
# ==========================================

def clean_2025(df):
    """2025년 데이터의 컬럼을 확인하고 공통 형식으로 변환한다."""

    print("\n2025년 데이터 컬럼:")
    print(df.columns.tolist())

    # 실제 컬럼 구성을 확인하기 전까지는
    # 임의로 장소나 좌표를 지정하지 않는다.
    print("2025년 데이터는 컬럼 확인 후 별도 전처리가 필요합니다.")

    return pd.DataFrame(
        columns=[
            "위도", "경도", "장소",
            "발생일", "시간대", "출처"
        ]
    )


# ==========================================
# 6. 장소별 발생 건수 집계
# ==========================================

def count_locations(df):
    """장소별 발생 건수를 계산한다."""

    return (
        df.groupby(["출처", "장소"], dropna=False)
        .size()
        .reset_index(name="발생건수")
        .sort_values("발생건수", ascending=False)
    )


# ==========================================
# 7. 지도 생성
# ==========================================

def create_map(df):
    """좌표가 있는 데이터를 지도에 표시한다."""

    # 좌표가 있는 행만 지도에 표시
    map_data = df.dropna(subset=["위도", "경도"]).copy()

    if map_data.empty:
        print("지도에 표시할 좌표 데이터가 없습니다.")
        return

    # 대구 북구 중심 좌표
    m = folium.Map(
        location=[35.885, 128.582],
        zoom_start=13,
        tiles="OpenStreetMap"
    )

    marker_cluster = MarkerCluster().add_to(m)

    # 장소별 발생 건수 계산
    counts = (
        map_data.groupby(
            ["출처", "장소", "위도", "경도"],
            dropna=False
        )
        .size()
        .reset_index(name="발생건수")
    )

    for _, row in counts.iterrows():
        popup_text = f"""
        <b>장소:</b> {row['장소']}<br>
        <b>출처:</b> {row['출처']}<br>
        <b>발생 건수:</b> {row['발생건수']}건
        """

        folium.CircleMarker(
            location=[row["위도"], row["경도"]],
            radius=min(5 + row["발생건수"] ** 0.5, 20),
            popup=folium.Popup(popup_text, max_width=300),
            tooltip=f"{row['장소']} ({row['발생건수']}건)",
            fill=True,
            fill_opacity=0.7
        ).add_to(marker_cluster)

    m.save("대구북구_불법주정차_지도.html")

    print("\n지도 저장 완료!")
    print("파일명: 대구북구_불법주정차_지도.html")


# ==========================================
# 8. 전체 실행
# ==========================================

def main():

    # 2020년 데이터
    df_2020 = load_csv(FILE_2020)
    data_2020 = clean_2020(df_2020)

    # 2024년 데이터
    df_2024 = load_csv(FILE_2024)
    data_2024 = clean_2024(df_2024)

    # 2025년 데이터
    df_2025 = load_csv(FILE_2025)
    data_2025 = clean_2025(df_2025)

    # 공통 형식의 데이터 통합
    all_data = pd.concat(
        [data_2020, data_2024, data_2025],
        ignore_index=True
    )

    # 통합 데이터 저장
    all_data.to_csv(
        "통합_불법주정차_데이터.csv",
        index=False,
        encoding="utf-8-sig"
    )

    # 장소별 발생 건수 확인
    location_counts = count_locations(all_data)

    location_counts.to_csv(
        "장소별_불법주정차_건수.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n장소별 발생 건수 상위 10개")
    print(location_counts.head(10))

    # 지도 생성
    create_map(all_data)


if __name__ == "__main__":
    main()

# 실행결과

# 정상적으로 진행시 총 파일 3개가 만들어짐


# | `통합_불법주정차_데이터.csv`   | 각 연도 데이터를 공통 형식으로 정리한 파일 |
# | `장소별_불법주정차_건수.csv`   | 장소별 발생 건수를 집계한 파일        |
# | `대구북구_불법주정차_지도.html` | 좌표가 있는 데이터를 지도에 표시한 파일   | --> 인터넷 브라우저로 오픈


######## 1. 통합_불법주정차_데이터.csv

# 목적: 여러 연도의 데이터를 하나의 표로 통합하는 파일

# 2020년, 2024년, 2025년 데이터를 공통된 컬럼 형식으로 정리한 파일

# 예시

# 위도	경도	장소	발생일	시간대	출처
# 35.89	128.58	침산동	2020-07-01	주간	2020년
# 35.88	128.59	복현동	2020-07-02	야간	2020년
# NaN	NaN	동천동 회전교차로1	2023-12-01	주간	2024년

######### 2. 장소별_불법주정차_건수.csv

# 목적: 장소별 불법 주정차 발생 건수를 집계하는 파일

# 첫 번째 파일이 개별 발생 기록을 담고 있다면, 두 번째 파일은 장소별로 발생 건수를 합산한 결과
# 예를 들어 다음과 같은 형태

# 출처	장소	발생건수
# 2020년	침산동	350
# 2020년	복현동	280
# 2024년	동천동 회전교차로1	2

######### 3. 대구북구_불법주정차_지도.html

# 목적: 불법 주정차 발생 위치를 지도에 시각화하는 파일

# 세 파일 중 실제로 지도에서 데이터를 확인할 수 있는 파일이야.
# HTML 파일을 웹 브라우저에서 열면 대구 북구를 중심으로 지도가 표시


[parking_2020.csv] 불러오기 성공: utf-8
데이터 크기: (67949, 5)
컬럼: ['경도', '위도', '행정동', '위반상태', '기준일자']

[parking_violation_2024.csv] 불러오기 성공: cp949
데이터 크기: (10555, 6)
컬럼: ['구분', '위반일자', '위반시간', '위반장소', '과태료부과일자', '데이터기준일자']

[parking_violation_2026.csv] 불러오기 성공: utf-8
데이터 크기: (12241, 5)
컬럼: ['���ݻ���', '�����Ͻ�', '�ܼӱ���', '���·�ΰ�����', '�����ͱ�������']

2025년 데이터 컬럼:
['���ݻ���', '�����Ͻ�', '�ܼӱ���', '���·�ΰ�����', '�����ͱ�������']
2025년 데이터는 컬럼 확인 후 별도 전처리가 필요합니다.

장소별 발생 건수 상위 10개
       출처     장소  발생건수
6   2020년    노원동  9222
9   2020년  무태조야동  6124
4   2020년    구암동  5016
5   2020년    국우동  4379
13  2020년   산격2동  4296
2   2020년    관문동  3482
16  2020년    읍내동  3304
8   2020년    동천동  3090
3   2020년    관음동  3037
22  2020년   태전2동  2922

지도 저장 완료!
파일명: 대구북구_불법주정차_지도.html
